In [ ]:
# 1. GEREKLİ TÜM PAKETLERİN TEK SEFERDE KURULUMU
!pip install -q transformers bitsandbytes accelerate chromadb sentence-transformers feedparser beautifulsoup4 langchain-text-splitters requests numpy

import torch
import os
from google.colab import userdata
from sentence_transformers import SentenceTransformer
import chromadb

print("🏗️ Sistem Altyapısı Başlatılıyor...")

print("🧬 Gelişmiş Çok Dilli Embedding Modeli (multilingual-e5-large) belleğe alınıyor...")
# Daha güçlü ve anlamsal bağı daha iyi kuran e5-large modeli (Adım 4)
embedding_model = SentenceTransformer("intfloat/multilingual-e5-large")

# 3. KALICI VEKTÖR HAFIZASI (ChromaDB Persistent Client)
print("💾 ChromaDB Veritabanı diske bağlanıyor...")
chroma_client = chromadb.PersistentClient(path="./chroma_news_db")

# Koleksiyonu al veya oluştur (Cosine Similarity ile)
# Yeni modelin vektör boyutu (1024) eski modelden (384) farklı olduğu için yeni bir koleksiyon oluşturuyoruz.
collection_name = "mynet_news_e5_large"
news_collection = chroma_client.get_or_create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)

print("\n" + "="*50)
print("✅ KATMAN 1 BAŞARIYLA ÇALIŞTI: Kütüphaneler kuruldu (ChromaDB vektör saklamak için, SentenceTransformers metni anlamsal uzaya çevirmek için).")
print("➡️ Sıradaki Adım: Dış dünyadan verileri toplayacak RSS bağlantılarının ve hata sigortalarının (Katman 2) hazırlanması.")
print("="*50)

In [ ]:
import time
import threading
from datetime import datetime
import feedparser
import requests
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np

print("⚙️ Katman 2: Ultra-Modüler Veri Motoru Başlatılıyor...")

# Katman 3'ün (Sorgu Ünitesi) kullanacağı global merkez havuzu
category_centroids = {}

# 1. KONSOLİDE HEDEFLER VE SİGORTALAR
CONSOLIDATED_FEEDS = {
    "Finans": ["https://finans.mynet.com/haber/rss/gununozeti/"],
    "Gündem": [
        "https://www.mynet.com/haber/rss/sondakika",
        "https://www.mynet.com/haber/rss/gununozeti/",
        "https://www.mynet.com/haber/rss/kategori/guncel/",
        "https://www.mynet.com/haber/rss/kategori/politika/",
        "https://www.mynet.com/haber/rss/kategori/dunya/",
        "https://www.mynet.com/haber/rss/kategori/yasam/"
    ],
    "Teknoloji": ["https://www.mynet.com/haber/rss/kategori/teknoloji/"],
    "Spor": [
        "https://www.mynet.com/spor/rss",
        "https://www.mynet.com/spor/rss?cat=Football"
    ],
    "Sağlık": ["https://www.mynet.com/haber/rss/kategori/saglik/"]
}

FALLBACK_ANCHORS = {
    "Finans": "Dolar, euro, altın fiyatları, borsa, hisse senedi, merkez bankası faiz kararı, enflasyon, piyasalar ve finans haberleri.",
    "Gündem": "Türkiye gündemi, son dakika haberleri, siyaset, meclis, bakanlık açıklamaları, iç politika ve toplumsal olaylar.",
    "Teknoloji": "Yapay zeka, akıllı telefonlar, sosyal medya, yazılım, uzay araştırmaları, siber güvenlik ve teknoloji dünyası.",
    "Spor": "Futbol, basketbol, süper lig maç sonuçları, transfer haberleri, şampiyonlar ligi, milli takım ve spor dünyası.",
    "Sağlık": "Hastane, doktor, tedavi yöntemleri, aşı, beslenme, psikoloji, iklim değişikliği ve sağlıklı yaşam rehberi."
}

print("\n✅ İŞLEM TAMAM: Botun nerelerden haber okuyacağı (RSS linkleri) ve internet koparsa uydurma yapmaması için acil durum çapaları (Fallback) tanımlandı.")
print("➡️ Sıradaki Adım: Bu RSS linklerine gidip ham veriyi çekecek fonksiyonun (Modül A) kodlanması.")

In [ ]:
# --- MODÜL A: HAM VERİ HASADI ---
def fetch_raw_feeds(category_urls, max_entries=15):
    """Belirtilen RSS hatlarına gidip ham başlık, link ve tarih verilerini toplar."""
    raw_articles = []
    for feed_url in category_urls:
        try:
            feed = feedparser.parse(feed_url)
            source_name = feed.feed.title if 'title' in feed.feed else "Mynet Haber"

            for entry in feed.entries[:max_entries]:
                raw_articles.append({
                    "title": entry.title,
                    "link": entry.link,
                    "date": entry.get('published', datetime.now().strftime("%d.%m.%Y | %H:%M")),
                    "summary": entry.summary if 'summary' in entry else "",
                    "source": source_name
                })
        except Exception:
            pass
    return raw_articles

print("✅ Modül A Hazır: Bu fonksiyon, webdeki RSS'leri tarayarak temel URL ve başlıkları liste halinde toplama görevini üstlenecek.")
print("➡️ Sıradaki Adım: Toplanan bu linklerin içine girip reklamları/HTML kodlarını silecek Modül B'nin tanımlanması.")

In [ ]:
# --- MODÜL B: İÇERİK TEMİZLEME VE KAZIMA ---
def clean_and_extract_text(article):
    """Linke giderek HTML etiketlerini temizler ve saf metni çıkarır."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
    }
    content = ""
    try:
        response = requests.get(article["link"], headers=headers, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, "html.parser")
            paragraphs = soup.find_all("p")
            content = " ".join([p.get_text(strip=True) for p in paragraphs])
    except Exception:
        pass

    # Sayfa boş dönerse RSS özetindeki metni kurtarma kalkanı
    if not content.strip() and article["summary"]:
        content = BeautifulSoup(article["summary"], "html.parser").get_text(separator=" ", strip=True)

    return content.strip()

print("✅ Modül B Hazır: Bu fonksiyon, BeautifulSoup kütüphanesi yardımıyla sitelerdeki gereksiz HTML'i silip, LLM'in okuyabileceği saf haber metnini elde edecek.")
print("➡️ Sıradaki Adım: Uzun haber metinlerini LLM limitine takılmamak için küçük parçalara (chunk) bölecek Modül C'nin tanımlanması.")

In [ ]:
# --- MODÜL C: METİN PARÇALAMA (CHUNKING) VE ID ATAMA ---
def chunk_documents(article, content, main_category, splitter):
    """Saf metni LLM uyumlu parçalara böler ve benzersiz kimlikler (ID) tanımlar."""
    chunks = splitter.split_text(content)
    processed_docs = []

    for idx, chunk_text in enumerate(chunks):
        # Deterministik ID: Aynı link tekrar geldiğinde veritabanında kopya oluşturmaz
        chunk_id = f"id_{main_category}_{abs(hash(article['link']))}_{idx}"
        enriched_text = f"Başlık: {article['title']} | İçerik: {chunk_text}"

        processed_docs.append({
            "id": chunk_id,
            "text": enriched_text,
            "metadata": {
                "title": article["title"],
                "source": article["source"],
                "url": article["link"],
                "timestamp": str(article["date"]),
                "category": main_category
            }
        })
    return processed_docs

print("✅ Modül C Hazır: Bu fonksiyon, metni parçalayarak ve onlara benzersiz bir ID atayarak veritabanında kopya oluşmasını engelleyecek.")
print("➡️ Sıradaki Adım: Bu parçalanmış metinleri makinenin anlayacağı matematiksel dizilere (vektörlere) dönüştürecek Modül D'nin tanımlanması.")

In [ ]:
# --- MODÜL D: DNA VEKTÖRLEŞTİRME ---
def generate_dna_vectors(docs):
    """Metin parçalarını Katman 1'deki modelle sayısal vektörlere dönüştürür."""
    if not docs:
        return []
    texts = [doc["text"] for doc in docs]
    # embedding_model Katman 1'den kalıcı olarak miras alınmıştır
    vectors = embedding_model.encode(texts).tolist()

    for i, doc in enumerate(docs):
        doc["embedding"] = vectors[i]
    return docs

In [ ]:
# --- MODÜL E: VERİTABANI YAZIMI VE MERKEZ (CENTROID) GÜNCELLEMESİ ---
def update_database_and_centroids(all_category_docs):
    """Hazır vektörleri ChromaDB'ye işler ve 5 ana merkezin konumunu sabitler. Çift ID'leri temizler."""
    global category_centroids
    total_upserts = 0

    print("\n📊 --- KONSOLİDE MERKEZ (CENTROID) RAPORU ---")
    for cat, docs in all_category_docs.items():
        if docs:
            # Aynı ID'ye sahip chunkları filtreleyerek tekilleştirme (Deduplication)
            unique_docs = {}
            for d in docs:
                if d["id"] not in unique_docs:
                    unique_docs[d["id"]] = d
            docs = list(unique_docs.values())

            ids = [d["id"] for d in docs]
            vecs = [d["embedding"] for d in docs]
            metas = [d["metadata"] for d in docs]
            texts = [d["text"] for d in docs]

            # Kalıcı veritabanına Upsert (Ekle veya Güncelle)
            news_collection.upsert(ids=ids, embeddings=vecs, metadatas=metas, documents=texts)

            # Ağırlık merkezinin (Mean) hesaplanması
            category_centroids[cat] = np.mean(vecs, axis=0).tolist()
            total_upserts += len(docs)
            print(f"✅ Küme: {cat:<10} | Aktif Chunk: {len(docs):<4} | Durum: MERKEZ GÜNCEL")
        else:
            print(f"⚠️ Küme: {cat:<10} | Aktif Chunk: 0    | Durum: ESKİ MERKEZ KORUNDU")

    print(f"🚀 Montaj Hattı Bitti! Veritabanında {total_upserts} adet taze chunk güvende.")


In [ ]:
# --- ORKESTRATÖR: OTONOM MONTAJ HATTI ---
def run_autonomous_pipeline():
    """Tüm modülleri sırasıyla birbirine bağlayan ana denetleyici akış."""
    print(f"\n🔄 [OTONOM HASAT - {datetime.now().strftime('%H:%M:%S')}] Modüler zincir tetiklendi...")
    splitter = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=50, length_function=len)
    all_category_docs = {cat: [] for cat in CONSOLIDATED_FEEDS.keys()}

    for main_category, urls in CONSOLIDATED_FEEDS.items():
        print(f"\n📂 --- KATEGORİ İŞLENİYOR: {main_category.upper()} ---")
        category_docs = []

        # 1. Adım: Ham Makaleleri Çek (Modül A)
        print("  📡 1. Adım: Ham RSS haberleri çekiliyor...")
        raw_articles = fetch_raw_feeds(urls)
        print(f"  ✅ Bulunan makale sayısı: {len(raw_articles)}")

        print("  🧼 & ✂️ 2-3. Adımlar: İçerik temizleniyor ve chunk'lara (parçalara) bölünüyor...")
        for article in raw_articles:
            # 2. Adım: İçeriği Temizle (Modül B)
            content = clean_and_extract_text(article)
            if not content:
                continue

            # 3. Adım: Chunklara Böl ve ID Ata (Modül C)
            docs = chunk_documents(article, content, main_category, splitter)
            category_docs.extend(docs)

        print(f"  ✅ {main_category} için toplam {len(category_docs)} adet metin parçası (chunk) üretildi.")

        # SİGORTA DEVREDE: Kategori tamamen boşsa Acil Durum Çapası işlenir
        if not category_docs:
            print(f"  🛡️ UYARI: '{main_category}' kümesi boş! Semantik Çapa (Fallback) devreye giriyor...")
            anchor_text = f"Başlık: {main_category} Acil Durum Çapası | İçerik: {FALLBACK_ANCHORS[main_category]}"
            category_docs.append({
                "id": f"fallback_{main_category}",
                "text": anchor_text,
                "metadata": {
                    "title": f"{main_category} Acil Durum Çapası", "source": "Sistem DNA Sigortası",
                    "url": "https://www.mynet.com", "timestamp": datetime.now().strftime("%d.%m.%Y | %H:%M"),
                    "category": main_category
                }
            })

        # 4. Adım: Vektörleri Üret (Modül D)
        print("  🧬 4. Adım: Metin parçaları yapay zeka vektörlerine (embedding) dönüştürülüyor...")
        vectorized_docs = generate_dna_vectors(category_docs)
        all_category_docs[main_category].extend(vectorized_docs)
        print("  ✅ Vektörleştirme tamamlandı.")

    # 5. Adım: Veritabanını ve Merkezleri Güncelle (Modül E)
    print("\n💾 5. Adım: Tüm veriler ChromaDB'ye yazılıyor ve küme merkezleri güncelleniyor...")
    update_database_and_centroids(all_category_docs)


In [ ]:
import time
import threading

# --- ARKA PLAN İŞ PARÇACIĞI (25 DAKİKALIK DÖNGÜ) ---
def start_daemon_engine():
    """Arka planda otonom veri toplama sürecini yönetir."""
    # Botun hemen konuşabilmesi için ilk akışı anında tetikliyoruz
    run_autonomous_pipeline()

    # Eğer BM25 indeksi Modül F'de tanımlandıysa onu da baştan yenileyelim
    if 'build_bm25_index' in globals():
        build_bm25_index()

    # İstemci-Sunucu mimarisine geçildiği için veritabanı kilitlenme riski minimize edildi.
    # Otonom döngüyü aktif ediyoruz:
    def loop():
        while True:
            time.sleep(1500)  # Tam 25 dakika (1500 saniye) bekleme aralığı
            print("\n⏰ [OTONOM ZAMANLAYICI] 25 dakika doldu. Yeni haberler taranıyor...")
            run_autonomous_pipeline()

            # Her yeni haber hasadından sonra BM25 kelime motorunu tazele
            if 'build_bm25_index' in globals():
                print("🔄 BM25 Kelime İndeksi güncelleniyor...")
                build_bm25_index()

    # Daemon thread (Arka plan iş parçacığı) olarak başlatıyoruz ki ana akışı (Gradio vb.) bloklamasın
    engine_thread = threading.Thread(target=loop, daemon=True)
    engine_thread.start()
    print("\n✅ OTONOM MOTOR AKTİF: Sistem her 25 dakikada bir kendini güncelleyecek.")

# Motoru Başlat
start_daemon_engine()


In [ ]:
!pip install -q rank_bm25 sentence-transformers

from rank_bm25 import BM25Okapi
import string
from sentence_transformers import CrossEncoder

print("⚙️ Modül F: İleri Düzey Hibrit Arama (BM25 + Vektör + Cross-Encoder) Hazırlanıyor...")

# Çok dilli Reranker (Yeniden Sıralayıcı) modeli RAM'e alınıyor
print("🧬 Cross-Encoder modeli yükleniyor (Bu işlem biraz sürebilir)...")
reranker = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')

# Global BM25 Değişkenleri
bm25 = None
all_docs = []
all_ids = []
all_metadatas = []

# Metinleri küçük harfe çevirip noktalama işaretlerinden arındıran basit bir tokenizasyon
def tokenize_text(text):
    if not text: return []
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    return text.split()

def build_bm25_index():
    """ChromaDB'deki son belgeleri çekip BM25 motorunu (yeniden) eğitir."""
    global bm25, all_docs, all_ids, all_metadatas
    db_data = news_collection.get()
    all_docs = db_data['documents']
    all_ids = db_data['ids']
    all_metadatas = db_data['metadatas']

    tokenized_corpus = [tokenize_text(doc) for doc in all_docs]
    bm25 = BM25Okapi(tokenized_corpus)
    print(f"✅ BM25 Anahtar Kelime Motoru {len(all_docs)} parça (chunk) ile indekslendi.")

# 1. BM25 İndeksini ilk kez oluşturalım
build_bm25_index()

def hybrid_search(query, top_k=3, category_filter=None):
    """Hem Vektör hem de BM25 skorlarını harmanlayan ve Cross-Encoder ile son puanlamayı yapan fonksiyon"""
    filtre_metni = f" [Filtre: {category_filter}]" if category_filter and category_filter != "Tümü" else ""
    print(f"\n🔍 Gelişmiş Hibrit Tarama Yapılıyor: '{query}'{filtre_metni}")

    # --- ADIM A: BM25 ARAMASI (Anahtar Kelime Bazlı) ---
    tokenized_query = tokenize_text(query)
    bm25_scores = bm25.get_scores(tokenized_query)

    scored_indices = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)
    if category_filter and category_filter != "Tümü":
        bm25_top_indices = [i for i in scored_indices if all_metadatas[i].get('category') == category_filter][:top_k*4]
    else:
        bm25_top_indices = scored_indices[:top_k*4]

    # --- ADIM B: VEKTÖR ARAMASI (Anlam Bazlı) ---
    query_vec = embedding_model.encode(query).tolist()
    where_clause = {"category": category_filter} if category_filter and category_filter != "Tümü" else None

    if where_clause:
        vector_results = news_collection.query(query_embeddings=[query_vec], n_results=top_k*4, where=where_clause)
    else:
        vector_results = news_collection.query(query_embeddings=[query_vec], n_results=top_k*4)

    vector_top_ids = vector_results['ids'][0] if vector_results['ids'] else []

    # --- ADIM C: RRF (RECIPROCAL RANK FUSION) BİRLEŞTİRME ---
    rrf_scores = {}

    for rank, idx in enumerate(bm25_top_indices):
        doc_id = all_ids[idx]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + (1 / (rank + 60))

    for rank, doc_id in enumerate(vector_top_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + (1 / (rank + 60))

    # RRF ile en iyi adayları bulalım (Cross-Encoder için havuz)
    sorted_rrf = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:15]

    # --- ADIM D: CROSS-ENCODER RE-RANKING (YENİDEN SIRALAMA) ---
    candidate_docs = []
    candidate_pairs = []

    for doc_id, score in sorted_rrf:
        idx = all_ids.index(doc_id)
        doc_text = all_docs[idx]
        candidate_docs.append({
            "id": doc_id,
            "text": doc_text,
            "metadata": all_metadatas[idx]
        })
        candidate_pairs.append([query, doc_text])

    if candidate_pairs:
        # Cross-Encoder modeline [Soru, Cevap_Adayı] çiftlerini verip gerçek uyumu ölçüyoruz
        cross_scores = reranker.predict(candidate_pairs)

        for i in range(len(candidate_docs)):
            candidate_docs[i]["cross_score"] = float(cross_scores[i])

        # Yeni skorlara göre en yüksekten en düşüğe sırala
        candidate_docs = sorted(candidate_docs, key=lambda x: x["cross_score"], reverse=True)

        # --- YENİ: BAŞLIK TEKİLLEŞTİRME (DEDUPLICATION) ---
        # Eşik (Threshold) kısıtlaması kaldırıldı! Genel sorularda bile haberlerin geçmesine izin veriliyor.
        final_docs = []
        seen_titles = set()

        for doc in candidate_docs:
            title = doc["metadata"]["title"]

            # Sadece kopya başlıkları engelliyoruz, düşük skorlu da olsa haberleri alıyoruz.
            if title not in seen_titles:
                final_docs.append(doc)
                seen_titles.add(title)

            # İstenen sayıya (top_k) ulaştıysak döngüyü bitir
            if len(final_docs) == top_k:
                break

        candidate_docs = final_docs

    return candidate_docs

# --- TEST EDELİM ---
ornek_soru = "Dolar bugün ne kadar oldu?"
sonuclar = hybrid_search(ornek_soru, category_filter="Finans")

print("\n🏆 EN İYİ RE-RANKED SONUÇLAR (Çeşitlendirilmiş):")
for i, res in enumerate(sonuclar):
    print(f"{i+1}. BAŞLIK: {res['metadata']['title']} | SKOR: {res.get('cross_score', 0):.2f}")


In [ ]:
import google.generativeai as genai
from google.colab import userdata

# --- GEMINI RAG TEST SİMÜLASYONU ---
print("🧪 Gemini RAG Üretim Testi Başlatılıyor...\n")

try:
    # 1. API Anahtarını al ve Gemini'yi yapılandır
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    gemini_model = genai.GenerativeModel('gemini-2.5-flash')
    print("✅ Gemini API bağlantısı kuruldu.")

    # 2. Test Sorusunu Belirle
    test_sorusu = "Galatasaray şampiyonluk kupasını ne zaman alacak?"
    print(f"\n❓ Soru: {test_sorusu}")

    # 3. Hibrit Arama (Modül F'yi kullanarak bağlamı/haberleri bul)
    arama_sonuclari = hybrid_search(test_sorusu, top_k=3)

    # 4. Prompt (Bağlam) Hazırlığı
    context_metni = ""
    for i, sonuc in enumerate(arama_sonuclari):
        context_metni += f"[Kaynak {i+1}]: {sonuc['text']}\n"

    prompt = f"""Sen Mynet Haber asistanısın. Aşağıdaki kaynakları kullanarak soruyu profesyonelce yanıtla.
Eğer kaynaklarda cevaba dair net bir bilgi yoksa, bunu kibarca belirt.

KAYNAKLAR:
{context_metni}

SORU: {test_sorusu}"""

    # 5. Gemini'den Cevap Üretmesini İste
    print("\n🤖 Gemini Düşünüyor (Metin Sentezleniyor)...\n")
    response = gemini_model.generate_content(prompt)

    # 6. Sonucu Ekrana Bas
    print("💬 GEMINI'NIN YANITI:")
    print("═"*50)
    print(response.text)
    print("═"*50)

except Exception as e:
    print(f"\n❌ Bir hata oluştu. Lütfen GOOGLE_API_KEY'in Secrets (🔑) kısmında ekli olduğundan emin olun.\nHata Detayı: {e}")


In [ ]:
# MODÜL G: SOHBET HAFIZASI VE RAG ASİSTANI (GEMINI API TARAFINDAN YÖNETİLİYOR)
# Bu hücredeki Gemma 3 tabanlı RAG Asistanı devre dışı bırakılmıştır.
# Sohbet hafızası ve RAG yanıtları artık doğrudan Katman 4'teki FastAPI üzerinden Gemini API tarafından sağlanmaktadır.

# --- Global Sohbet Hafızası, FastAPI Modülü tarafından yönetilecektir. ---

In [ ]:
!pip install -q -U google-generativeai

In [ ]:
# 1. API BAĞIMLILIKLARININ KURULUMU
!pip install -q fastapi uvicorn pydantic nest-asyncio pyngrok google-generativeai

import nest_asyncio
import uvicorn
import threading
import socket
import google.generativeai as genai
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok
from google.colab import userdata
import time
import traceback
import os

# --- PORT VE THREAD TEMİZLİĞİ (Gecikme ve Yanıtsızlık Çözümü) ---
# Arka planda asılı kalan ve 8000 portunu işgal eden eski sunucuları öldürür.
os.system("fuser -k 8000/tcp")
time.sleep(2) # Kapanması için kısa bir süre tanı

# --- GLOBAL YAPILANDIRMA VE KALKANLAR ---
nest_asyncio.apply()
chat_histories = {}
if 'OTONOM_THREAD_CALISIYOR' not in globals():
    OTONOM_THREAD_CALISIYOR = False

# --- ZEKA YAPILANDIRMASI ---
try:
    gemini_key = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=gemini_key)
    gemini_engine = genai.GenerativeModel('gemini-2.5-flash')
    print("✅ Gemini API Motoru Hazır.")
except Exception as e:
    print(f"⚠️ Gemini API Key hatası: {e}")
    gemini_engine = None

# --- API ŞEMALARI ---
class QueryRequest(BaseModel):
    query: str
    session_id: str = "default"

class QueryResponse(BaseModel):
    status: str
    response: str
    engine: str

# --- SORGULAMA YÖNLENDİRİCİSİ (QUERY ROUTER) ---
def detect_category(query: str):
    """Kullanıcı sorusundan hedef kategoriyi tahmin eder (Hızlı Yönlendirme)."""
    q = query.lower()
    if any(w in q for w in ["ekonomi", "finans", "dolar", "euro", "altın", "borsa", "para", "faiz", "piyasa", "kredi", "enflasyon", "merkez bankası", "hisse", "kripto", "bitcoin", "maaş", "zam"]): return "Finans"
    if any(w in q for w in ["gündem", "siyaset", "politika", "meclis", "bakan", "cumhurbaşkanı", "seçim", "parti", "chp", "akp", "mhp", "sondakika", "haber", "türkiye", "polis", "mahkeme", "cinayet"]): return "Gündem"
    if any(w in q for w in ["spor", "futbol", "basketbol", "transfer", "maç", "galatasaray", "fenerbahçe", "beşiktaş", "trabzonspor", "şampiyon", "lig", "derbi", "hakem", "tff", "milli takım", "voleybol", "olimpiyat"]): return "Spor"
    if any(w in q for w in ["teknoloji", "yapay zeka", "telefon", "yazılım", "apple", "google", "uzay", "bilim", "robot", "siber", "internet", "microsoft", "oyun", "sosyal medya", "twitter", "instagram"]): return "Teknoloji"
    if any(w in q for w in ["sağlık", "hastalık", "doktor", "hastane", "tedavi", "beslenme", "diyet", "kanser", "virüs", "salgın", "ilaç", "aşı", "psikoloji", "kilo"]): return "Sağlık"
    return "Tümü" # Varsayılan olarak tüm veritabanında ara

# --- FASTAPI UYGULAMASI ---
app = FastAPI(title="Mynet Intelligence API", version="3.1.0")

def get_hybrid_response(user_query: str, session_id: str = "default"):
    """RAG Arama ve Gemini Sentezleme Mantığı"""
    global chat_histories
    active_engine = "Gemini-2.5-Flash"

    # Veri yapısını dict olarak güncelliyoruz (Özet + Geçmiş)
    if session_id not in chat_histories or isinstance(chat_histories[session_id], list):
        chat_histories[session_id] = {"summary": "", "history": []}

    session_data = chat_histories[session_id]

    # Hibrit arama
    try:
        context_blocks = ""
        unique_sources = {}

        if 'hybrid_search' in globals():
            # Kategori Tespiti ve Yönlendirmeli Arama
            detected_cat = detect_category(user_query)

            # --- YENİ: DİNAMİK ARAMA DERİNLİĞİ ---
            # Soru genel bir haber özeti istiyorsa (örn: neler oldu, haberler) 6 kaynak getir, spesifikse 3 kaynak.
            genel_soru_kaliplari = ["neler", "son dakika", "haber", "özet", "gündem", "bugün"]
            is_general = any(kalip in user_query.lower() for kalip in genel_soru_kaliplari)
            dinamik_top_k = 6 if is_general else 3

            results = hybrid_search(user_query, top_k=dinamik_top_k, category_filter=detected_cat)

            if results:
                for i, res in enumerate(results):
                    meta = res['metadata']
                    context_blocks += f"\n[KAYNAK {i+1}]\nBAŞLIK: {meta['title']}\nİÇERİK: {res['text']}\nLİNK: {meta['url']}\n"
                    unique_sources[meta['url']] = meta
            else:
                context_blocks = "İlgili haber bulunamadı."
        else:
            context_blocks = "Arama motoru başlatılamadı."

        # --- ADIM 3: GELİŞMİŞ SOHBET HAFIZASI YÖNETİMİ (MEMORY SUMMARIZATION) ---
        # Eğer geçmiş çok şiştiyse eski konuşmaları LLM ile özetle
        if len(session_data["history"]) > 3:
            old_chats = session_data["history"][:-2]
            text_to_summarize = session_data["summary"] + "\n"
            for c in old_chats:
                text_to_summarize += f"Kullanıcı: {c['user']}\nBot: {c['bot'][:200]}...\n"

            try:
                summary_prompt = f"Sen bir sohbet özetleyicisin. Aşağıdaki kullanıcı ve bot arasındaki sohbet geçmişinin ana konusunu ve önemli detaylarını tek bir kısa paragrafta özetle:\n{text_to_summarize}"
                new_summary = gemini_engine.generate_content(summary_prompt).text
                session_data["summary"] = new_summary
                session_data["history"] = session_data["history"][-2:] # Sadece son 2'yi tut
            except Exception as e:
                print(f"Özetleme hatası: {e}")

        # Hafıza Dizgesini (String) Hazırla
        history_str = ""
        if session_data["summary"]:
            history_str += f"[ÖNCEKİ SOHBETLERİN ÖZETİ: {session_data['summary']}]\n\n"

        for chat in session_data["history"]:
            history_str += f"Kullanıcı: {chat['user']}\nBot: {chat['bot'].split('📚')[0]}\n"

        # GELİŞMİŞ VE PROFESYONEL PROMPT MÜHENDİSLİĞİ
        prompt = f"""Sen üst düzey profesyonel bir Mynet Haber asistanısın. Aşağıdaki kaynakları kullanarak kullanıcıya yapılandırılmış, okunması kolay ve estetik bir yanıt ver.

KURALLAR:
1. Yanıtını Markdown formatında ver (Başlıklar, kalın yazılar, listeler kullan).
2. Eğer kullanıcı genel bir soru soruyorsa (örneğin "gündemde ne var", "siyasi haberler neler" gibi), kaynaklarda spesifik bir cevap aramaya çalışma. Bunun yerine, sağlanan kaynaklardaki haberleri derleyerek genel bir özet sun.
3. Kaynaklardaki bilgileri kullanarak kullanıcının niyetine en uygun cevabı oluştur.
4. Okunabilirliği yüksek, tarafsız ve net bir dil kullan.
5. Kaynaklarda kesinlikle sorulan konuyla veya özetlenecek genel haberle alakalı hiçbir bilgi yoksa, durumu kibarca belirt. Kesinlikle uydurma (halüsinasyon) yapma.
6. ÇOK ÖNEMLİ: Yukarıdaki 'ÖNCEKİ SOHBET GEÇMİŞİ' sadece konunun bağlamını anlaman içindir. Cevabını KESİNLİKLE aşağıda verilen güncel 'KAYNAKLAR' kısmına göre oluştur. Eski bilgileri tekrarlama, daima taze kaynaklara odaklan!

{history_str}

KAYNAKLAR:
{context_blocks}

SORU: {user_query}"""

        response = gemini_engine.generate_content(prompt)
        answer = response.text

        # Kaynakları programatik ve estetik (Markdown Link) olarak ekle
        if unique_sources:
            answer += "\n\n---\n### 📚 İlgili Kaynaklar\n"
            for i, m in enumerate(unique_sources.values(), 1):
                tarih = m.get('timestamp', 'Tarih belirtilmemiş')
                answer += f"* [{m['title']}]({m['url']}) - *({tarih})*\n"

        # Yeni mesajı geçmişe ekle
        session_data["history"].append({"user": user_query, "bot": answer})

        return answer, active_engine
    except Exception as e:
        return f"Hata oluştu: {str(e)}", "Error"

# --- ENDPOINTLER ---
@app.get("/")
@app.get("/health")
@app.get("/system-health")
def health_check():
    return {"status": "online", "engine": "Gemini-2.5-Flash", "db_status": "Connected"}

@app.post("/ask", response_model=QueryResponse)
def api_ask(request: QueryRequest):
    # API Çağrısı
    answer, engine = get_hybrid_response(request.query, request.session_id)
    return QueryResponse(status="success", response=answer, engine=engine)

# --- SUNUCU VE TÜNEL ---
PORT = 8000
ngrok.kill()
try:
    token = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(token)
    public_url = ngrok.connect(PORT).public_url
    print(f"\n🚀 STREAMLIT İÇİN URL: {public_url}\n")
except:
    print("⚠️ ngrok başlatılamadı!")

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

threading.Thread(target=run_api, daemon=True).start()


In [ ]:
# Gradio Kütüphanesinin Kurulumu
!pip install -q gradio

In [ ]:
import gradio as gr
import requests

print("⚙️ Gradio Arayüzü Hazırlanıyor...")

# Gerçek İstemci-Sunucu (Client-Server) İletişimi
def chat_with_rag(message, history):
    url = "http://127.0.0.1:8000/ask"
    payload = {
        "query": message,
        "session_id": "gradio_session"
    }

    try:
        response = requests.post(url, json=payload, timeout=30)
        if response.status_code == 200:
            data = response.json()
            return data.get("response", "Sunucudan boş yanıt geldi.")
        else:
            return f"⚠️ Sunucu Hatası: {response.status_code} - {response.text}"

    except requests.exceptions.ConnectionError:
        return "❌ API sunucusuna ulaşılamıyor. Lütfen FastAPI hücresinin (Modül G) çalıştığından emin olun."
    except requests.exceptions.Timeout:
        return "⏳ API sunucusu yanıt vermekte gecikti (Zaman Aşımı)."
    except Exception as e:
        return f"❌ Beklenmeyen bir hata oluştu: {str(e)}"

# --- ARAYÜZ GÜZELLEŞTİRMELERİ ---
# Çok daha şık, premium bir tema (Turkuaz/Mavi tonları ve Montserrat fontu)
custom_theme = gr.themes.Ocean(
    primary_hue="teal",
    secondary_hue="blue",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Montserrat"), "ui-sans-serif", "system-ui", "sans-serif"]
).set(
    body_background_fill="*neutral_50",
    block_radius="12px",
    block_border_width="0px",
    block_shadow="*shadow_drop_lg",
    button_primary_background_fill="*primary_600",
    button_primary_background_fill_hover="*primary_500",
    button_primary_text_color="white",
)

# Gradio ChatInterface oluşturma
demo = gr.ChatInterface(
    fn=chat_with_rag,
    title="🌐 Mynet Akıllı Haber Merkezi",
    description="### Yapay Zeka Destekli Kesintisiz Haber Asistanı\nTürkiye ve dünyadaki gelişmeleri anında sentezleyen, **Gemini 2.5 Flash** ve **Hibrit RAG** mimarisiyle güçlendirilmiş haber motoru.",
    examples=[
        "Gündemdeki en önemli siyasi haberler neler?",
        "Teknoloji ve yapay zeka dünyasında bugün ne oldu?",
        "Son dakika spor ve transfer gelişmeleri neler?"
    ],
    theme=custom_theme,
    chatbot=gr.Chatbot(
        show_label=False,
        avatar_images=(None, "https://cdn-icons-png.flaticon.com/512/6146/6146825.png"), # Daha modern bir haber botu ikonu
        height=600
    ),
    textbox=gr.Textbox(placeholder="Örn: Ekonomi dünyasında altın fiyatları ne durumda?", container=False, scale=7)
)

demo.launch(share=True, debug=True)
